# Imports

In [1]:
from __future__ import annotations

import json
import requests

import pandas as pd

from pathlib import Path
from typing import Any

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

# Fixed variables

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DATA_DIR = DATA_DIR / "input"

DATA_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://apidatos.ree.es/es/datos"

CATEGORY = "demanda"
WIDGET = "evolucion"

START_DATE = "2026-01-01T00:00"
END_DATE = "2026-06-30T23:59"

# periods = [
#     # ("2023-01-01T00:00", "2023-06-30T23:59"),
#     # ("2023-07-01T00:00", "2023-12-31T23:59"),
#     # ("2024-01-01T00:00", "2024-06-30T23:59"),
#     # ("2024-07-01T00:00", "2024-12-31T23:59"),
#     ("2025-01-01T00:00", "2025-06-30T23:59"),
#     ("2025-07-01T00:00", "2025-12-31T23:59"),
#     ("2026-01-01T00:00", "2026-06-30T23:59"),
# ]

TIME_TRUNC = "day"


PERIODS_PATH = INPUT_DATA_DIR / "periods.json"
output_path = INPUT_DATA_DIR / "01_redes.csv"

# API Call

In [3]:
with open(
    PERIODS_PATH,
    encoding="utf-8",
) as file:
    config = json.load(file)

periods = config["periods"]

In [4]:
endpoint = f"{BASE_URL}/{CATEGORY}/{WIDGET}"

params = {
    "start_date": START_DATE,
    "end_date": END_DATE,
    "time_trunc": TIME_TRUNC,
}

dfs = []

for start_date, end_date in periods:

    params["start_date"] = start_date
    params["end_date"] = end_date

    response = requests.get(
        endpoint,
        params=params,
        timeout=30,
    )

    print(
        start_date,
        end_date,
        response.status_code,
    )

    response.raise_for_status()

    data = response.json()

    df_batch = pd.DataFrame(
        data["included"][0]["attributes"]["values"]
    )

    dfs.append(df_batch)

2025-01-01T00:00:00UTC 2025-06-30T23:59:59UTC 200


2025-07-01T00:00:00UTC 2025-12-31T23:59:59UTC 200


2026-01-01T00:00:00UTC 2026-06-30T23:59:59UTC 200


In [5]:
df_redes = pd.concat(
    dfs,
    ignore_index=True,
)

# Save Data

In [6]:
df_redes.to_csv(output_path, index=False)

print(f"Dataset saved to:\n{output_path}")

Dataset saved to:
C:\Users\JuanOrtizAlonso\spain-electricity-demand-forecasting\data\input\01_redes.csv
